# <DOMAIN> domain — Qwen3-8B LoRA finetune (Unsloth)

Template notebook on `main`. Each `domain/<name>` branch keeps its own copy of this
file tailored to that domain (dataset field names, DOMAIN value below) — see
`docs/ADDING_A_DOMAIN.md`.

Run top to bottom on a Colab GPU runtime (Runtime → Change runtime type → T4 GPU,
or A100 on Colab Pro). Writes `domains/<name>/results/{baseline,finetuned}.json`
and `domains/<name>/adapters/final/`. Commit those back when done — see the last cell.

In [ ]:
DOMAIN = "<name>"  # e.g. "code", "finance" — must match domains/<DOMAIN>/

In [ ]:
# Unsloth's exact install command changes with Colab's CUDA/torch version —
# check https://github.com/unslothai/unsloth for the current recommended cell
# if this one fails.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q evalplus datasets huggingface_hub pyyaml

In [ ]:
import os

REPO_URL = "https://github.com/rchhabra13/llm-finetune-lab.git"

if not os.path.exists("llm-finetune-lab"):
    !git clone -b domain/$DOMAIN $REPO_URL
%cd llm-finetune-lab

## 1. Data prep

Download → exact/near-dup dedup → decontaminate against HumanEval/MBPP. Same scripts you can run locally without a GPU — see `docs/ARCHITECTURE.md`.

In [ ]:
!python -m src.data.download --config domains/$DOMAIN/data_config.yaml
!python -m src.data.dedup --config domains/$DOMAIN/data_config.yaml
!python -m src.data.decontaminate --config domains/$DOMAIN/data_config.yaml

## 2. Baseline eval

Score the stock base model *before* touching the weights — this is what "finetuned" gets compared against. Runs against the base model repo id directly, no adapter.

In [ ]:
import yaml

base_cfg = yaml.safe_load(open("configs/base.yaml"))
base_model = base_cfg["base_model"]
baseline_out = f"domains/{DOMAIN}/results/baseline.json"

!python -m src.eval.run_eval --model-path {base_model} --dataset humaneval --out {baseline_out}
!python -m src.eval.run_eval --model-path {base_model} --dataset mbpp --out {baseline_out}

## 3. Load model + attach LoRA

In [ ]:
from unsloth import FastLanguageModel

lora_cfg = base_cfg["lora"]
training_cfg = base_cfg["training"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=base_cfg["max_seq_length"],
    load_in_4bit=base_cfg["load_in_4bit"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["alpha"],
    lora_dropout=lora_cfg["dropout"],
    target_modules=lora_cfg["target_modules"],
    bias=lora_cfg["bias"],
    use_gradient_checkpointing="unsloth",
    random_state=training_cfg["seed"],
)

## 4. Format dataset

Assumes a two-field instruction dataset: `data_config.yaml`'s `text_fields` list is
`[input_field, output_field]`. If your domain dataset isn't a simple input/output
pair (e.g. multi-turn), edit `to_chat` below instead of forcing it through this shape.

In [ ]:
import json
from datasets import Dataset

data_cfg = yaml.safe_load(open(f"domains/{DOMAIN}/data_config.yaml"))
input_field, output_field = data_cfg["text_fields"][0], data_cfg["text_fields"][-1]

records = [json.loads(l) for l in open(f"domains/{DOMAIN}/data/processed/train.jsonl")]

def to_chat(example):
    messages = [
        {"role": "user", "content": example[input_field]},
        {"role": "assistant", "content": example[output_field]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(records).map(to_chat)
print(dataset)

## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=base_cfg["max_seq_length"],
    args=SFTConfig(
        per_device_train_batch_size=training_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=training_cfg["gradient_accumulation_steps"],
        num_train_epochs=training_cfg["num_train_epochs"],
        learning_rate=training_cfg["learning_rate"],
        lr_scheduler_type=training_cfg["lr_scheduler_type"],
        warmup_ratio=training_cfg["warmup_ratio"],
        weight_decay=training_cfg["weight_decay"],
        optim=training_cfg["optim"],
        seed=training_cfg["seed"],
        output_dir=f"domains/{DOMAIN}/adapters/checkpoints",
        logging_steps=10,
        save_strategy="no",
    ),
)

trainer_stats = trainer.train()

In [ ]:
adapter_dir = f"domains/{DOMAIN}/adapters/final"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

## 6. Finetuned eval

Same harness, same seed, same benchmark as the baseline cell above — the diff between the two is the real result.

In [ ]:
finetuned_out = f"domains/{DOMAIN}/results/finetuned.json"

!python -m src.eval.run_eval --model-path {base_model} --adapter-path {adapter_dir} --dataset humaneval --out {finetuned_out}
!python -m src.eval.run_eval --model-path {base_model} --adapter-path {adapter_dir} --dataset mbpp --out {finetuned_out}

In [ ]:
!python -m src.eval.report --domain $DOMAIN

## 7. Commit results back

```bash
git add domains/$DOMAIN/results domains/$DOMAIN/adapters/final
git commit -m "$DOMAIN domain: baseline vs finetuned eval results"
git push
```

Then update `docs/RESULTS.md` and the domain status table in the root `README.md` on `main`.